# 03 · Capa plata: modelo relacional en tercera forma normal (3FN)

**Qué hace:** lee `bronze_contratos` y construye **10 tablas** limpias y relacionadas,
verifica en los datos las dependencias funcionales en que se basa el diseño, y comprueba
llaves primarias e integridad referencial.

| Tabla | Llave primaria | Contenido |
|---|---|---|
| `dim_departamento` | `departamento` | Departamento y su país |
| `dim_municipio` | `departamento` + `municipio` | Municipios (llave compuesta: hay nombres repetidos entre departamentos) |
| `dim_entidad` | `codigo_entidad` | Entidad compradora; ubicación como FK (departamento, municipio) |
| `dim_proveedor` | `codigo_proveedor` | Código interno, tipo de documento y si es grupo. **Sin datos personales** |
| `dim_familia_unspsc` | `familia_unspsc` | Familia UNSPSC y su segmento |
| `dim_categoria` | `codigo_categoria` | Código UNSPSC y FK a su familia |
| `dim_proceso` | `proceso_de_compra` | Modalidad y justificación (son del proceso, no del contrato) |
| `contrato` | `id_contrato` | Lo que se sabe **al firmar** |
| `contrato_fuente_recurso` | `id_contrato` + `fuente` | Fuentes de recursos en formato largo |
| `contrato_ejecucion` | `id_contrato` | Lo que pasa **después** de firmar (estado, pagos, días añadidos) |

**Formas normales:**
- **1FN** (valores atómicos, sin grupos repetidos): la localización "País, Departamento,
  Ciudad" se separa; "6 Mes(es)" pasa a número y unidad; las 6 columnas de fuentes de
  recursos pasan a filas.
- **2FN** (con llave compuesta, todo depende de la llave completa): en
  `contrato_fuente_recurso`, el valor depende del contrato **y** de la fuente; `dim_municipio`
  no tiene atributos que dependan solo del departamento (el país está en `dim_departamento`).
- **3FN** (sin dependencias transitivas): la modalidad depende del proceso, no del contrato;
  el segmento depende de la familia, no del código; el país depende del departamento.
  Tampoco se guardan datos calculables (los días de duración se calculan en oro).

Toda la lógica está en `src/secop/plata.py` (probada con pytest). Aquí solo se lee,
se llama a las funciones, se escribe y se verifica.

## Configuración: importar `src/secop` y crear la sesión de Spark

1. Buscamos la carpeta del repositorio (la que contiene `src/secop`) subiendo desde la
   carpeta del notebook y añadimos `src` a `sys.path`, la lista de carpetas donde Python
   busca módulos. Así `import secop` funciona sin instalar nada.
2. `entorno.crear_spark()` crea una sesión de **Spark en modo local** con Delta Lake: el
   mismo motor que en un clúster, usando los núcleos del Codespace.

Si cambias código en `src/`, reinicia el kernel (botón *Restart*) para que Python lo
vuelva a cargar.

In [ ]:
import sys
from pathlib import Path


def buscar_raiz_repo() -> Path:
    """Devuelve la carpeta raíz del repo (la que contiene src/secop)."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "src" / "secop").is_dir():
            return carpeta
    raise RuntimeError("No encontré src/secop: abre el notebook desde la carpeta del repositorio.")


RAIZ_REPO = buscar_raiz_repo()
if str(RAIZ_REPO / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ_REPO / "src"))

import secop
from secop import config, entorno

spark = entorno.crear_spark()
print("secop", secop.__version__, "| Spark", spark.version, "| datos en", config.raiz_datos())

## 1. Leer bronce
`leer_tabla` no lee datos todavía: crea un **plan**. Spark es *perezoso* (lazy): las
transformaciones se acumulan y solo se ejecutan al llegar a una **acción** (escribir,
contar, mostrar).

In [ ]:
from pyspark.sql import functions as F

from secop import plata
from secop.validacion import exigir_sin_personales

bronce = entorno.leer_tabla(spark, config.TABLA_BRONCE)

## 2. Dependencias funcionales: ¿los datos cumplen lo que supone el diseño?

Normalizar se basa en **dependencias funcionales**: "X → Y" significa que cada valor de X
tiene un único valor de Y (por ejemplo, cada proceso de compra tiene una sola modalidad).
Aquí se cuenta, en los datos reales, cuántos valores de X aparecen con más de un Y.

- **0 violaciones:** la dependencia se cumple y la tabla está bien separada.
- **Más de 0:** hay datos inconsistentes en la fuente. La dimensión los resuelve con una
  regla explícita (el nombre más largo o el valor más reciente). Anotar el número en
  `docs/informe.md`. Referencia: en 2019, "entidad → nombre" tenía 8 casos (nombre y sigla).

In [ ]:
entorno.mostrar(plata.revisar_dependencias(bronce))

## 3. Construir y escribir las 10 tablas

Cada tabla se reemplaza completa (es pequeña y así el resultado no depende de
ejecuciones anteriores) y acepta cambios de esquema. Se escriben en orden: primero las
tablas padre.

No usamos `cache()`: cada tabla se recalcula desde bronce, que con ~500 mil filas es
rápido, y una vez escrita se lee desde Delta.

In [ ]:
for nombre, df in plata.construir_plata(bronce).items():
    exigir_sin_personales(df.columns)  # ninguna tabla de plata puede tener datos personales
    entorno.escribir_tabla(df, nombre)
    print("escrita:", nombre)

# Desde aquí se trabaja con las tablas YA escritas (no con los planes en memoria).
t = {nombre: entorno.leer_tabla(spark, nombre) for nombre in plata.CONSTRUCTORES}

## 4. Llaves primarias: únicas y sin nulos
Una llave primaria identifica cada fila: no puede repetirse ni estar vacía.

In [ ]:
unicidad = []
for tabla_nombre, llave in plata.LLAVES_PRIMARIAS.items():
    nulos = F.lit(False)
    for c in llave:
        nulos = nulos | F.col(c).isNull()
    fila = (
        t[tabla_nombre]
        .agg(
            F.count("*").alias("filas"),
            F.countDistinct(*llave).alias("llaves"),
            F.sum(nulos.cast("long")).alias("nulos"),
        )
        .first()
    )
    unicidad.append(
        (tabla_nombre, "+".join(llave), fila["filas"], fila["llaves"], fila["nulos"] or 0)
    )

entorno.mostrar(
    spark.createDataFrame(
        unicidad, "tabla string, llave string, filas long, llaves_distintas long, llaves_nulas long"
    )
)
assert all(f == k and n == 0 for _, _, f, k, n in unicidad), "Llave primaria duplicada o nula"
print("Llaves primarias: OK")

## 5. Integridad referencial
Cada llave foránea debe existir en su tabla padre (incluida la llave compuesta
departamento + municipio). `left_anti` devuelve las filas del hijo **sin** pareja en el
padre: el conteo debe ser **0**. Si no lo es, el notebook se detiene (mejor fallar aquí
que entrenar con datos rotos).

In [ ]:
resultados = []
for hijo, padre, llave in plata.RELACIONES:
    huerfanas = plata.filas_huerfanas(t[hijo], t[padre], llave).count()
    texto_llave = llave if isinstance(llave, str) else "+".join(llave)
    resultados.append((hijo, padre, texto_llave, huerfanas))

entorno.mostrar(
    spark.createDataFrame(resultados, "hijo string, padre string, llave string, huerfanas long")
)
rotas = [r for r in resultados if r[3] != 0]
assert not rotas, f"Integridad referencial rota: {rotas}"
print(f"Integridad referencial: OK (0 huérfanas en las {len(resultados)} relaciones)")

## 6. Casos que resolvió la normalización
- Entidades que tenían más de un nombre (en 2019 se observaron 8: nombre completo y sigla).
- Procesos con más de una modalidad en la fuente (si hay, se usó la más reciente).
- Municipios con el mismo nombre en varios departamentos (justifican la llave compuesta).

In [ ]:
entorno.mostrar(
    t["dim_entidad"].filter("n_nombres > 1").select("codigo_entidad", "nombre_entidad", "n_nombres")
)
entorno.mostrar(t["dim_proceso"].filter("n_modalidades > 1"))
entorno.mostrar(
    t["dim_municipio"]
    .groupBy("municipio")
    .agg(F.count("*").alias("departamentos"))
    .filter("departamentos > 1")
    .orderBy(F.desc("departamentos"))
)

## 7. Vistazo a las limpiezas
Estado unificado (ya sin mayúsculas inconsistentes) y duración separada en número y unidad.

In [ ]:
entorno.mostrar(t["contrato_ejecucion"].groupBy("estado_contrato").count().orderBy(F.desc("count")))
entorno.mostrar(
    t["contrato"]
    .groupBy("duracion_unidad")
    .agg(F.count("*").alias("contratos"), F.avg("duracion_numero").alias("numero_promedio"))
)